## RAG 第 3 天

### InsureLLM 專家問答系統

LangChain 1.0 實作的 RAG 管線。

使用上次建立的 VectorStore（HuggingFace `all-MiniLM-L6-v2`）

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [ ]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

### 連接 Chroma；使用 Hugging Face all-MiniLM-L6-v2

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### 設定兩個關鍵 LangChain 物件：retriever 與 llm

#### 關於「temperature」的補充：
- 控制輸出多樣性
- temperature 為 0 表示輸出應可預測
- 較高 temperature 可得到更多樣的答案

有人把 temperature 比作「創意」，但不完全正確
- 它實際控制推論時選哪些 token
- temperature=0：總是選機率最高的 token
- temperature=1：通常表示 10% 機率的 token 有 10% 機會被選中

注意：temperature 為 0 不代表輸出一定可重現。還需設定隨機種子。我們會在 week 6–8 處理。（即便如此，有時仍無法完全重現。）

注意 2：若要創意，請用 System Prompt！

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### 這些 LangChain 物件都實作 `invoke()` 方法

In [ ]:
retriever.invoke("Who is Avery?")

In [ ]:
llm.invoke("Who is Avery?")

## 是時候組裝起來了！

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

## 接下來還能玩什麼？😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## 承認吧——你以為 RAG 會更複雜！！